# 01 — Data Quality Check
Kiểm tra schema, null values, FK integrity, date coverage cho 14 tables.

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.data_loader import load_all_tables

tables = load_all_tables()
print("Tables loaded:", list(tables.keys()))

Tables loaded: ['sales', 'sample_submission', 'shipments', 'web_traffic', 'customers', 'geography', 'inventory', 'order_items', 'orders', 'payments', 'products', 'promotions', 'returns', 'reviews']


## 1. Schema & Shape

In [2]:
for name, df in tables.items():
    print(f'\n=== {name} === ({df.shape[0]:,} rows x {df.shape[1]} cols)')
    print(df.dtypes.to_string())


=== sales === (3,833 rows x 3 cols)
Date       datetime64[us]
Revenue           float64
COGS              float64

=== sample_submission === (548 rows x 3 cols)
Date       datetime64[us]
Revenue           float64
COGS              float64

=== shipments === (566,067 rows x 4 cols)
order_id                  int64
ship_date        datetime64[us]
delivery_date    datetime64[us]
shipping_fee            float64

=== web_traffic === (3,652 rows x 7 cols)
date                        datetime64[us]
sessions                             int64
unique_visitors                      int64
page_views                           int64
bounce_rate                        float64
avg_session_duration_sec           float64
traffic_source                         str

=== customers === (121,930 rows x 7 cols)
customer_id                     int64
zip                             int64
city                              str
signup_date            datetime64[us]
gender                       category
age_group   

## 2. Null Analysis

In [10]:
for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        pct = (nulls / len(df) * 100).round(2)
        print(f'\n{name}:')
        print(pct.to_string())


order_items:
promo_id      61.34
promo_id_2    99.97

promotions:
applicable_category    80.0


## 3. FK Integrity

In [4]:
checks = [
    ('orders',      'customer_id', 'customers', 'customer_id'),
    ('orders',      'zip',         'geography', 'zip'),
    ('order_items', 'order_id',    'orders',    'order_id'),
    ('order_items', 'product_id',  'products',  'product_id'),
    ('payments',    'order_id',    'orders',    'order_id'),
    ('shipments',   'order_id',    'orders',    'order_id'),
    ('returns',     'order_id',    'orders',    'order_id'),
    ('returns',     'product_id',  'products',  'product_id'),
    ('reviews',     'order_id',    'orders',    'order_id'),
    ('inventory',   'product_id',  'products',  'product_id'),
]
for child, fk, parent, pk in checks:
    orphans = ~tables[child][fk].isin(tables[parent][pk])
    print(f'{child}.{fk} -> {parent}.{pk}: {orphans.sum()} orphans')

orders.customer_id -> customers.customer_id: 0 orphans
orders.zip -> geography.zip: 0 orphans
order_items.order_id -> orders.order_id: 0 orphans
order_items.product_id -> products.product_id: 0 orphans
payments.order_id -> orders.order_id: 0 orphans
shipments.order_id -> orders.order_id: 0 orphans
returns.order_id -> orders.order_id: 0 orphans
returns.product_id -> products.product_id: 0 orphans
reviews.order_id -> orders.order_id: 0 orphans
inventory.product_id -> products.product_id: 0 orphans


## 4. Sales Date Coverage & Gaps

In [ ]:
sales = tables['sales'].sort_values('Date')
print(f"Train range : {sales.Date.min()} to {sales.Date.max()}")
print(f"Total days  : {len(sales)}")
full_range = pd.date_range(sales.Date.min(), sales.Date.max())
missing = full_range.difference(sales.Date)
print(f"Missing days: {len(missing)}")
if len(missing):
    print(missing.tolist())

Train range : 2012-07-04 00:00:00 to 2022-12-31 00:00:00
Total days  : 3833
Missing days: 0


: 

## 5. Business Rules Validation

In [6]:
# cogs < price
products = tables['products']
v = products[products['cogs'] >= products['price']]
print(f'cogs >= price violations: {len(v)}')

# Revenue > 0 and COGS > 0
sales = tables['sales']
print(f'Revenue <= 0: {(sales.Revenue <= 0).sum()}')
print(f'COGS <= 0   : {(sales.COGS <= 0).sum()}')

cogs >= price violations: 0
Revenue <= 0: 0
COGS <= 0   : 0


## 6. Duplicate PKs

In [7]:
pk_map = {
    'products': ['product_id'],
    'customers': ['customer_id'],
    'geography': ['zip'],
    'orders': ['order_id'],
    'payments': ['order_id'],
    'sales': ['Date'],
}
for tbl, keys in pk_map.items():
    dupes = tables[tbl].duplicated(subset=keys).sum()
    print(f'{tbl}: {dupes} duplicate PKs')

products: 0 duplicate PKs
customers: 0 duplicate PKs
geography: 0 duplicate PKs
orders: 0 duplicate PKs
payments: 0 duplicate PKs
sales: 0 duplicate PKs
